# Lesson 05 - Agentic RAG


## Setup

Dis notebook dey show how Agentic RAG (Retrieval-Augmented Generation) pattern dey work wit di Microsoft Agent Framework.

**Wetin you go need:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment wey you don set wit environment variables
- Azure CLI wey you don log in (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Wetin be Agentic RAG?

Traditional RAG dey follow one kain fixed pipeline: dem go collect documents, then dem go generate answer. **Agentic RAG** pass this one, e dey give the agent freedom to choose **when** and **how** to collect information.

Wit Agentic RAG, the agent fit:
- **Decide** if e need collect information before e answer question
- **Choose** which data source or tool e go use ask
- **Evaluate** the information wey e collect and fit collect more if the first one no clear
- **Combine** information from plenty retrieval steps make e form correct answer

Dis one make the agent flexible wella and accurate pass the static retrieve-then-generate pipeline.


## How to Create Search Tool

For Agentic RAG, outside data sources dem na **tools** wey agent fit use anytime e want. Dis one make agent fit see retrieval as another action wey e fit do, no be say e be compulsory step.

Below, we go define travel knowledge base and make e show say na tool wey agent fit call to find destination information.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Di Bilt RAG Agent

Now we go create one agent wey dem tell make e **always find tori first before e answer**. Di agent go use di `search_travel_knowledge` tool to base im answer on top di knowledge base instead of just to rely on im own training data.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — The Maker-Checker Pattern

Big benefit wey Agentic RAG get na **iterative retrieval**. Di agent fit do plenty rounds of search to confirm, correct, or add more info to wetin e first find — e just be like "maker-checker" work waka:

1. **Maker step**: Di agent go find di first info come draft answer.
2. **Checker step**: Di agent go do extra search to double check details or complete wetin miss.

Under, dem ask di agent question wey need am to compare plenty places, so e go search many many times.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

For dis lekshan, you don learn how to build **Agentic RAG** system using Microsoft Agent Framework:

- **Agentic RAG** dey make agents fit decide by demself when dem suppose find information, wey make di retrieval dey dynamic, no be fixed.
- **Tools as data sources**: External knowledge bases (like Azure AI Search) dem wrap as tools wey agent fit call.
- **Iterative retrieval**: Maker-checker pattern go help agent run multiple retrieval rounds — search, verify, and improve — before e give final answer.

For production, you go change di in-memory `TRAVEL_KNOWLEDGE_BASE` put real Azure AI Search index to handle big big travel document retrieval.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
